# Evolutionäre Algorithmen – Optimierung durch Selektion, Rekombination und Mutation

## Szenario: Maximum einer Funktion finden

In diesem Notebook verwenden wir einen **Evolutionären Algorithmus**, um das Maximum der Funktion

$
f(x_1,x_2)=1.5+\cos(0.3\,x_1x_2)
$

numerisch zu suchen.

Anders als bei klassischen Optimierungsverfahren wird dabei nicht nur eine einzelne Lösung betrachtet. Stattdessen arbeitet der Algorithmus mit einer **Population von Lösungskandidaten**, die über mehrere Generationen verändert und bewertet wird.

Dabei verwenden wir typische Bestandteile evolutionärer Algorithmen:

- eine **Population** von Individuen,
- eine **Fitnessfunktion**,
- **Selektion**,
- **Crossover/Rekombination**,
- **Mutation**,
- und die Übernahme besonders guter Lösungen durch **Elitismus**.

> **Wichtig:** Ein evolutionärer Algorithmus garantiert im Allgemeinen nicht, dass er das globale Optimum findet. Durch zufällige Variation und Selektion kann er sich jedoch schrittweise in Richtung guter Lösungen bewegen.

## Lernziele

1. das Grundprinzip eines **Evolutionären Algorithmus** erklären, zwischen **Individuum, Population, Genom und Fitness** unterscheiden, 

2. erläutern, warum Fitnesswerte für bestimmte Selektionsverfahren gegebenenfalls verschoben werden müssen,

3. die Rolle von **Elitismus** verstehen,

4. einfache **Crossover- und Mutationsoperatoren** implementieren,

5. erklären, wie Selektion, Rekombination und Mutation gemeinsam neue Generationen erzeugen,

6. den Verlauf der besten Fitness über mehrere Generationen interpretieren.

## Ablauf

Das Notebook gliedert sich in vier Schritte:

**1. Optimierungsproblem verstehen**  
Bestimmen Sie zunächst analytisch, welchen maximalen Wert die Zielfunktion annehmen kann.

**2. Evolutionären Algorithmus analysieren**  
Untersuchen Sie Population, Fitnessfunktion, Selektion und Elitismus in der gegebenen Implementierung.

**3. Variation implementieren**  
Implementieren Sie die beiden zentralen Operatoren **Crossover** und **Mutation**.

**4. Optimierung durchführen**  
Starten Sie den evolutionären Algorithmus und beobachten Sie, wie sich die Qualität der gefundenen Lösungen über die Generationen entwickelt.


# 1. Optimierungsproblem verstehen

Im Folgenden nutzen wir einen evolutionären Algorithmus, um das Maximum der Funktion

$cos(0.3*x_1*x_2) + 1.5$

zu bestimmen. Bevor wir starten: Was ist das Maximum dieser Funktion? 

### TODO
Das Maximum ist 2,5
Die funktion wird maximal, wenn $x_1$ und $x_2$ = 0 sind. 


# 2. Evolutionären Algorithmus analysieren

Betrachten Sie nun die folgende Implementierung: 

In [4]:
# Based on an implementation by Soroush Oskouei 
import random
import math
import matplotlib.pyplot as plt
import numpy as np

def initialize_population(population_size, dims, lower_bound, upper_bound):
    """
    Initialize a population of size `population_size`.
    Each individual is represented by a list of length `dims`,
    with each value between `lower_bound` and `upper_bound`.
    """
    population = []
    for _ in range(population_size):
        individual = [random.uniform(lower_bound, upper_bound) for _ in range(dims)]
        population.append(individual)
    return population

def fitness_function(individual):
    # Our (hard-coded) goal is to maximize cos(0.3*x_1*x_2) + 1.5
    return 1.5 + math.cos(0.3*individual[0]*individual[1])

def shift_fitnesses(fitnesses):
    """
    Shift all fitness values so that they're strictly positive.
    This is needed for roulette wheel selection (to avoid negative or zero sum).
    """
    min_f = min(fitnesses)
    if min_f <= 0:
        shift_amount = abs(min_f) + 1e-6
        return [f + shift_amount for f in fitnesses]
    else:
        return fitnesses

def roulette_wheel_selection(population, fitnesses):
    """
    Fitness-proportionate selection, assuming `fitnesses` are >= 0.
    """
    total_fitness = sum(fitnesses)
    
    # Fallback if total_fitness is 0 or negative (should not happen if properly shifted)
    if total_fitness <= 0:
        return random.choice(population)
    
    pick = random.uniform(0, total_fitness)
    current = 0
    for i, individual in enumerate(population):
        current += fitnesses[i]
        if current >= pick:
            return individual
    # Fallback
    return population[-1]


def get_elites(population, fitnesses, num_elites=1):
    """
    Returns the top `num_elites` individuals based on fitness.
    Idea: Preserve so far best solutions without cross-iover or mutations
    """
    # Sort population by fitness in descending order
    # Note: key=lambda x: x[1] tells python to sort by the second element of 
    #       each tuple (i.e., the fitness value)
    sorted_pop = sorted(zip(population, fitnesses), key=lambda x: x[1], reverse=True)
    elites = [ind for ind, fit in sorted_pop[:num_elites]]
    return elites


def evolutionary_algorithm(population_size, dims, lower_bound, upper_bound, 
                           mutation_rate, generations, num_elites=1):
    """
    An evolutionary algorithm that runs for `generations` iterations and
    prints the best raw fitness every 10 generations.
    """
    # log for plotting
    best_fitness_per_gen = np.zeros(generations)
    
    # initialize population
    population = initialize_population(population_size, dims, lower_bound, upper_bound)
    
    for gen in range(generations):
        # evaluate raw fitness
        fits = [fitness_function(ind) for ind in population]
        
        # shift fitnesses for roulette wheel selection
        shifted_fits = shift_fitnesses(fits)
        
        # get elites
        elites = get_elites(population, shifted_fits, num_elites)
        
        new_population = []
        
        # generate new population (excluding elites for now)
        while len(new_population) < (population_size - num_elites):
            # Select parents
            parent1 = roulette_wheel_selection(population, shifted_fits)
            parent2 = roulette_wheel_selection(population, shifted_fits)
            
            # Crossover
            child1, child2 = crossover_swap(parent1, parent2)
            
            # Mutation
            child1 = mutation(child1, mutation_rate, lower_bound, upper_bound)
            child2 = mutation(child2, mutation_rate, lower_bound, upper_bound)
            
            new_population.append(child1)
            new_population.append(child2)
            
        # Add elites back in
        new_population.extend(elites)
        
        # Update population
        population = new_population
        
        # Print best raw fitness every 10 generations
        gen_best_raw_fit = max(fitness_function(ind) for ind in population)
        if (gen + 1) % 100 == 0:
            print(f"Generation {gen+1}, Best Raw Fitness: {gen_best_raw_fit:.4f}")
        best_fitness_per_gen[gen] = gen_best_raw_fit
    
    # After all generations, find the best individual
    final_raw_fits = [fitness_function(ind) for ind in population]
    best_index = max(range(len(population)), key=lambda i: final_raw_fits[i])
    best_individual = population[best_index]
    best_fit = final_raw_fits[best_index]
    
    plt.figure()
    plt.plot(np.linspace(1, generations, generations), best_fitness_per_gen)
    plt.xlabel("Generations")
    plt.ylabel("Fitness")
    
    return best_individual, best_fit


Welche Aufgaben haben die Funktionen

* `roulette_wheel_selection`

* `shift_fitnesses`

* `get_elites`

Was sind Ein- und Ausgaben?

### TODO

roulette_wheel_selection:
Eingaben: Population und fitnesses
Return: Gibt ein individuum zurück, welches an der stelle ist bei der alle fitness werte der vorherigen und des aktuellens als summe einen vorab zufällig gewählten wert ergeben.

shift_fitnesses:
Eingabe: fitnesses
Return: Wählt denn niedrigsten wert aus denn fitnesswerten aus, sollte eins negativ sein, schiebt er alle fitness werte so weit nachoben, dass keins negativ ist. Dabei bleibt der abstand aller gleich. 

get_elites: 
Eingabe: population, fitnesses, num_elites
Return: identifiziert die besten individuen basierend auf ihrem fittness wert.



# 3. Variation implementieren

Implementieren Sie nun die beiden Funktionen (Beschreibung siehe Code-Kommentare)

* `crossover_swap(parent1, parent2)`
* `mutation(individual, mutation_rate, lower_bound, upper_bound)`

Hinweis: Folgende Befehle könnten nützlich sein:

* `random.uniform`
* `random.randint`

In [6]:

def crossover_swap(parent1, parent2):
    """
    Perform a swap crossover in one dimension (i.e., one randomly chosen gene in the genome) for two parent individuals.
    Yields two children.
    
    # Example usage:
    p1 = [1, 2, 3, 4]
    p2 = [5, 6, 7, 8]
    c1, c2 = crossover_swap(p1, p2)
    print("Crossover result:", c1, c2)
    
    yields (for idx=1)
    p1 = [1, 6, 3, 4]
    p2 = [5, 2, 7, 8]
    """

    c_1 = parent1.copy()
    c_2 = parent2.copy()

    idx = random.randrange(len(parent1))

    c_1[idx], c_2[idx] = c_2[idx], c_1[idx]
    return c_1, c_2

def mutation(individual, mutation_rate, lower_bound, upper_bound):
    """
    For each element in the individual, with probability `mutation_rate`,
    replace it with another value within the bounds.
    """
    for i in range(len(individual)):
        if random.random() < mutation_rate:
            individual[i] = random.uniform(lower_bound, upper_bound)

    return individual
        



Sie können Ihre Lösung mittels folgendem Code überprüfen:

In [12]:
# Test crossover with example:
p1 = [1, 2, 3, 4]
p2 = [5, 6, 7, 8]
c1, c2 = crossover_swap(p1, p2)
print("Crossover result:", c1, c2)

# Mutation example
print("Child:", c1)
mutated_child = mutation(c1, 0.1, -10, 10)
print("Mutated child:", mutated_child)

Crossover result: [1, 2, 7, 4] [5, 6, 3, 8]
Child: [1, 2, 7, 4]
Mutated child: [1, 2, 7, -9.22679385975492]


# 4. Optimierung durchführen

Führen Sie nun die Optimierung durch, indem Sie folgenden Code ausführen.

In [ ]:
# Example usage:
if __name__ == "__main__":
    best_individual, best_fit = evolutionary_algorithm(
        population_size=20,
        dims=2,
        lower_bound=-10,
        upper_bound=10,
        mutation_rate=0.1,
        generations=100,    # Run for 1000 generations
        num_elites=1
    )
    
    print("\nBest individual found:", best_individual)
    print("Best (raw) fitness found:", best_fit)

*Quellenhinweis*: Bei der Formattierung der Einleitung dieses Notebooks wurde ChatGPT-5.1 verwendet.